# NLCD land-cover quiz: can you recognize land cover from its ALS point cloud?

Same idea as [vegbank_class_quiz.ipynb](vegbank_class_quiz.ipynb), but on the NLCD evaluation task in [datasets/NLCD_eval](../../datasets/NLCD_eval/README.md): analyst-interpreted NLCD 2021 reference pixels, each matched to a 3DEP lidar collection from the same year.

1. Pick a classification level (Level I, 8 classes, or Level II, 16 classes) and which classes to include.
2. The notebook shows a few **example 3D lidar tiles** for each class.
3. You are then **quizzed on a fresh tile** (never one of the examples) drawn from one of those classes.

The analysts also gave an **alternate label** where the class is ambiguous (about half the points). The quiz keeps a strict score (primary label only) and a lenient score (primary or alternate), which is a useful reference point for how well a probe on lidar embeddings could hope to do.

Tiles are streamed from the USGS 3DEP EPT on AWS, so the first draw of each tile takes a few seconds. Tiles are cached on disk, so re-running is fast.

## Configuration

In [1]:
LEVEL = 'nlcd_L1_class'   # 'nlcd_L1_class' (8 Level I classes) or 'nlcd_class' (16 Level II classes)
SELECTED_CLASSES = None   # list of class names to quiz on, or None to choose N_CLASSES automatically
N_CLASSES = 6             # used when SELECTED_CLASSES is None
CLASS_SELECTION = 'largest'  # 'random' or 'largest' (the N_CLASSES most common classes)
N_EXAMPLES = 3            # example tiles shown per class
RADIUS = 50               # tile radius in meters; the labeled pixel is the central 30 m
SUBSET = 'balanced'       # 'balanced' (class-balanced subset), 'stable' (label unchanged 2016-2021), or 'all'
SHOW_PIXEL = True         # outline the labeled 30 m pixel in red
SEED = None               # int for a reproducible quiz, None for a different one every run
MAX_POINTS = 15000        # points per tile are randomly thinned to this many for plotting
COLOR_BY = 'height'       # 'height', 'intensity', or a false-color blend: see false_color.MODES (printed below)
HEIGHT_MODE = 'above_ground'  # 'above_ground' (terrain removed, shows vegetation structure) or 'elevation' (raw Z, includes slope)
HEIGHT_CLIP = 10        # color scale max in m above lowest point; None = use the 99th percentile of shown tiles
TABLE = '../../datasets/NLCD_eval/nlcd_lidar_eval.parquet'
STATUS = '../../datasets/NLCD_eval/nlcd_lidar_eval_als_status.csv'  # streaming check; known-empty tiles are skipped
CACHE_DIR = '~/.cache/eagle_als_quiz'

In [2]:
import hashlib
import warnings
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from plotly.subplots import make_subplots
from IPython.display import display, clear_output
from tqdm.auto import tqdm

import false_color as fc
from dataset import load_als_cookie

rng = np.random.default_rng(SEED)
cache_dir = Path(CACHE_DIR).expanduser()
cache_dir.mkdir(parents=True, exist_ok=True)

## Load labeled points and choose classes

The alternate label is mapped to the same level as `LEVEL`, so at Level I the alternate counts only if it falls in a different Level I class. Points whose tile is known to be empty (from the streaming status file) are dropped up front.

In [3]:
nlcd = pd.read_parquet(TABLE)

# Level I class of the alternate label: Level I code is the Level II code rounded down to the tens
l1_names = dict(zip(nlcd['nlcd_L1_code'], nlcd['nlcd_L1_class']))
nlcd['alt_nlcd_L1_class'] = nlcd['alt_nlcd_code'].map(lambda c: l1_names.get(c // 10 * 10) if pd.notna(c) else None)
ALT = {'nlcd_class': 'alt_nlcd_class', 'nlcd_L1_class': 'alt_nlcd_L1_class'}[LEVEL]

if SUBSET == 'balanced':
    points = nlcd[nlcd['balanced_subset']]
elif SUBSET == 'stable':
    points = nlcd[nlcd['stable_2016_2021']]
else:
    points = nlcd
status = pd.read_csv(STATUS)
empty = set(status.loc[status['status'] == 'empty', 'als_site_id'])
points = points[~points['als_site_id'].isin(empty)].copy()


def class_label(c):
    return str(c)


counts = points[LEVEL].value_counts()
eligible = counts[counts > N_EXAMPLES].index   # need >N_EXAMPLES points so there is a fresh quiz tile left over
if SELECTED_CLASSES is not None:
    missing = [c for c in SELECTED_CLASSES if c not in eligible]
    if missing:
        raise ValueError(f"Not enough points for {missing}. Choose from: {list(eligible)}")
    candidates = list(SELECTED_CLASSES)
    N_CLASSES = len(candidates)
elif CLASS_SELECTION == 'largest':
    candidates = list(counts[eligible].index)
else:
    candidates = list(rng.permutation(eligible))
if len(eligible) < N_CLASSES:
    raise ValueError(f"Only {len(eligible)} {LEVEL} classes have more than {N_EXAMPLES} points; lower N_CLASSES or N_EXAMPLES.")
print(f"{len(points)} points; {len(eligible)} of {len(counts)} {LEVEL} classes have enough points:")
print(counts.to_string())

940 points; 8 of 8 nlcd_L1_class classes have enough points:
nlcd_L1_class
Developed             211
Forest                211
Planted/Cultivated    155
Wetlands              100
Herbaceous             97
Shrubland              94
Water                  40
Barren                 32


## Tile loading

`get_tile` streams a cylinder of radius `RADIUS` around the center of the reference pixel, returns heights relative to the lowest point, and caches the result. Points whose tile comes back empty (a known issue with some of the matched tiles) or fail to stream are skipped and another point is drawn. A red square at ground level marks the 30 m pixel the label applies to (set `SHOW_PIXEL = False` to hide it).

In [4]:
def height_above_ground(arr, max_ground=20000):
    """Z minus a ground surface interpolated from ASPRS class-2 points. NaN if there are too few ground points."""
    ground = arr[arr['Classification'] == 2]
    if len(ground) < 50:
        return np.full(len(arr), np.nan, dtype='float32')
    if len(ground) > max_ground:
        ground = ground[np.random.default_rng(0).choice(len(ground), max_ground, replace=False)]
    xy = np.column_stack([ground['X'], ground['Y']])
    q = np.column_stack([arr['X'], arr['Y']])
    surf = LinearNDInterpolator(xy, ground['Z'])(q)
    outside = np.isnan(surf)   # outside the ground-point hull: fall back to nearest ground point
    if outside.any():
        surf[outside] = NearestNDInterpolator(xy, ground['Z'])(q[outside])
    return (arr['Z'] - surf).astype('float32')


def get_tile(row):
    """Return DataFrame[X, Y, Z, HAG, Intensity, ReturnType] (meters, relative to the lowest point / ground) for a plot row, or None if no points."""
    key = hashlib.md5(f"nlcd_{row['point_id']}_{RADIUS}_v4".encode()).hexdigest()
    path = cache_dir / f"{key}.parquet"
    if path.exists():
        pts = pd.read_parquet(path)
        return pts if len(pts) >= 100 else None
    try:
        pipeline = load_als_cookie(
            als_tile_name=row['product_name_AWS'], lat=row['lat'], lon=row['lon'],
            point_crs='EPSG:4326', radius=RADIUS, out_crs=None,
        )
        arr = pipeline.arrays[0] if pipeline.arrays else []
    except Exception as e:  # network / EPT errors, point outside tile, ...
        warnings.warn(f"{row['point_id']}: {type(e).__name__}: {str(e)[:100]}")
        return None  # not cached, may work on retry
    if len(arr):
        pts = pd.DataFrame({k: np.asarray(arr[k] - arr[k].min(), dtype='float32') for k in ['X', 'Y', 'Z']})
        pts['Intensity'] = np.asarray(arr['Intensity'], dtype='float32')
        pts['ReturnType'] = fc.return_type_codes(arr['ReturnNumber'], arr['NumberOfReturns'])
        pts['HAG'] = height_above_ground(arr)
    else:
        pts = pd.DataFrame(columns=['X', 'Y', 'Z', 'HAG', 'Intensity', 'ReturnType'], dtype='float32')
    pts.to_parquet(path)
    return pts if len(pts) >= 100 else None


def draw_tiles(candidates, n, exclude=()):
    """Draw n loadable points from `candidates` (DataFrame), skipping excluded point_ids. Returns [(row, tile)]."""
    pool = candidates[~candidates['point_id'].isin(exclude)]
    pool = pool.iloc[rng.permutation(len(pool))]
    out = []
    for _, row in pool.iterrows():
        tile = get_tile(row)
        if tile is not None:
            out.append((row, tile))
        if len(out) == n:
            break
    return out


def heights(tile):
    """Height values to plot (m). Above-ground heights are clipped at 0; tiles with no ground points fall back to raw elevation."""
    if HEIGHT_MODE == 'above_ground' and tile['HAG'].notna().all():
        return tile['HAG'].clip(lower=0)
    return tile['Z']


def point_colors(tile, height_max, color_by):
    """Marker settings for one tile: dict(color, colorscale, cmin, cmax, showscale, colorbar)."""
    if color_by == 'intensity':
        i = tile['Intensity']
        lo, hi = np.percentile(i, [2, 98])   # clip outliers; scale is per tile
        return dict(color=i.clip(lo, hi), colorscale='Cividis', cmin=lo, cmax=hi, showscale=False)
    if color_by in fc.MODES:
        rgb = fc.false_color(heights(tile), tile['Intensity'], tile['ReturnType'], mode=color_by, hag_max=height_max)
        return dict(color=fc.to_plotly_colors(rgb), showscale=False)
    return dict(color=heights(tile), colorscale='Viridis', cmin=0, cmax=height_max, showscale=True,
                colorbar=dict(title='height (m)', len=0.6))


def thin(tile):
    return tile if len(tile) <= MAX_POINTS else tile.iloc[rng.choice(len(tile), MAX_POINTS, replace=False)]


def scene_fig(tiles, titles=None, height_max=None, width=330, color_by=None):
    """One row of 3D scenes, one per tile, sharing a height color scale and axis ranges.
    color_by: one name for all scenes, or a list with one per tile; defaults to COLOR_BY."""
    n = len(tiles)
    color_by = color_by or COLOR_BY
    color_by = [color_by] * n if isinstance(color_by, str) else color_by
    tiles = [thin(t) for t in tiles]
    height_max = height_max or HEIGHT_CLIP or max(np.percentile(heights(t), 99) for t in tiles)
    fig = make_subplots(rows=1, cols=n, specs=[[{'type': 'scene'}] * n], subplot_titles=titles,
                        horizontal_spacing=0.01)
    for i, t in enumerate(tiles, 1):
        fig.add_trace(go.Scatter3d(
            x=t['X'], y=t['Y'], z=heights(t), mode='markers',
            marker=dict(size=1.2, **{**point_colors(t, height_max, color_by[i - 1]),
                                     'showscale': (i == n and color_by[i - 1] == 'height')}),
            hoverinfo='skip',
        ), row=1, col=i)
        if SHOW_PIXEL:   # the label applies to this 30 m pixel; tiles are centered on it
            lo, hi = RADIUS - 15, RADIUS + 15
            fig.add_trace(go.Scatter3d(x=[lo, hi, hi, lo, lo], y=[lo, lo, hi, hi, lo], z=[0] * 5, mode='lines',
                                       line=dict(color='red', width=4), hoverinfo='skip'), row=1, col=i)
        # same box for all tiles so scale is comparable: x/y span the tile, z spans the tallest value shown
        fig.update_scenes(dict(
            xaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            yaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            zaxis=dict(range=[0, max(height_max, 2 * RADIUS * 0.4)], title='m'),
            aspectmode='manual', aspectratio=dict(x=1, y=1, z=max(height_max, 2 * RADIUS * 0.4) / (2 * RADIUS)),
        ), row=1, col=i)
    fig.update_layout(width=width * n + 60, height=360, margin=dict(l=0, r=0, t=40, b=0), showlegend=False)
    return fig

## Color options

`COLOR_BY` in the config cell takes `'height'`, `'intensity'`, or one of the false-color blends defined in [false_color.py](false_color.py), which combine height above ground, intensity and return type (only / first of many / intermediate / last of many) into one color. The next cell draws a single tile in every option so you can compare, and prints what each channel means.

In [5]:
for name, text in fc.DESCRIPTIONS.items():
    print(f"{name}: {text}")

demo_row, demo_tile = draw_tiles(points, 1)[0]
demo_tile = thin(demo_tile)   # thin once so every panel shows the same points
options = ['height', 'intensity'] + list(fc.MODES)
for k in range(0, len(options), 4):
    chunk = options[k:k + 4]
    scene_fig([demo_tile] * len(chunk), titles=chunk, color_by=chunk).show()

hsv_return_hue: hue = return type (only=blue, first=green, intermediate=yellow, last=magenta); saturation = intensity; value = height above ground
hsv_height_hue: hue = height above ground (blue low to red high); saturation = intensity; value = return type (only brightest, last-of-many darkest)
rgb_channels: red = height above ground; green = intensity; blue = return depth (first=0, intermediate=0.5, only=0.85, last=1)
ternary: color triangle: corners are height (teal), intensity (orange), return depth (violet); a point blends the corners by its values
palette_shaded: color = return type (only=grey, first=teal, intermediate=orange, last=violet); brightness = height above ground; vividness = intensity


## Example tiles

One row per class. Rotate the tiles with the mouse. The axes box is the same size for every tile, so canopy height is directly comparable. Each title gives the Level II class, so at Level I you can see which kinds of land cover a class lumps together.

In [ ]:
examples = {}   # class -> [(row, tile), ...]
with tqdm(total=N_CLASSES, desc='classes') as bar:
    for c in candidates:
        tiles = draw_tiles(points[points[LEVEL] == c], N_EXAMPLES + 1)   # +1: guarantees a loadable quiz tile exists
        if len(tiles) < N_EXAMPLES + 1:
            print(f"skipping {class_label(c)}: only {len(tiles)} loadable tiles")
            continue
        examples[c] = tiles[:N_EXAMPLES]
        bar.update(1)
        if len(examples) == N_CLASSES:
            break
if len(examples) < N_CLASSES:
    raise RuntimeError(f"Only found {len(examples)} usable classes; lower N_CLASSES or N_EXAMPLES")
classes = sorted(examples)
shown_ids = {r['point_id'] for tiles in examples.values() for r, _ in tiles}

In [ ]:
all_max = max(np.percentile(heights(t), 99) for tiles in examples.values() for _, t in tiles)
for c in classes:
    print(f"\n=== {class_label(c)} ({counts[c]} points) ===")
    scene_fig([t for _, t in examples[c]], titles=[r['nlcd_class'] for r, _ in examples[c]]).show()

## Quiz

Run the next cell. Pick the class you think the new tile belongs to. **Next tile** draws another one (the tile is always from one of the classes above and never one of the examples). The answer, the analysts' alternate label, and the location are revealed after you answer. A guess that matches only the alternate label counts toward the lenient score.

The last cell prints a confusion table of your answers.

In [ ]:
score = {'right': 0, 'lenient': 0, 'total': 0}
history = []   # (true class, guess) per answered tile
state = {}
out_fig = widgets.Output()
out_msg = widgets.HTML()
status_html = widgets.HTML()
choice_buttons = [widgets.Button(description=class_label(c), layout=widgets.Layout(width='auto'), tooltip=class_label(c))
                  for c in classes]
next_button = widgets.Button(description='Next tile', button_style='info', disabled=True)


def score_text():
    return f"<b>Score: {score['right']} / {score['total']}</b> (lenient: {score['lenient']} / {score['total']})"


def new_question(_=None):
    for b in choice_buttons + [next_button]:
        b.disabled = True
    out_msg.value = ''
    status_html.value = 'loading tile from AWS...'
    # pick the class first so every class is equally likely, regardless of how many points it has
    for _try in range(len(classes)):
        c = classes[rng.integers(len(classes))]
        drawn = draw_tiles(points[points[LEVEL] == c], 1, exclude=shown_ids | state.get('asked', set()))
        if drawn:
            break
    else:
        status_html.value = 'could not load a tile for any class'
        return
    row, tile = drawn[0]
    state.update(row=row, answer=c, answered=False)
    state.setdefault('asked', set()).add(row['point_id'])
    with out_fig:
        clear_output(wait=True)
        scene_fig([tile], titles=['Which class is this?'], height_max=all_max, width=700).show()
    status_html.value = score_text()
    for b in choice_buttons:
        b.disabled = False


def answer(button):
    if state.get('answered'):
        return
    state['answered'] = True
    correct = state['answer']
    row = state['row']
    alt = row[ALT]
    guess = classes[choice_buttons.index(button)]
    score['total'] += 1
    score['right'] += int(guess == correct)
    score['lenient'] += int(guess in (correct, alt))
    history.append((correct, guess))
    if guess == correct:
        verdict = '<span style="color:green"><b>Correct!</b></span>'
    elif guess == alt:
        verdict = f'<span style="color:orange"><b>Matches the alternate label.</b></span> You said {class_label(guess)}.'
    else:
        verdict = f'<span style="color:red"><b>Not quite.</b></span> You said {class_label(guess)}.'
    alt_text = f"alternate label: {row['alt_nlcd_class']}" if pd.notna(row['alt_nlcd_class']) else 'no alternate label'
    map_url = f"https://www.google.com/maps/search/?api=1&query={row['lat']:.6f},{row['lon']:.6f}"
    out_msg.value = (f"{verdict}<br>Answer: <b>{class_label(correct)}</b> ({row['nlcd_class']}; {alt_text})<br>"
                     f"Point {row['point_id']}, labeled {row['year']}, lidar {row['product_name_AWS']} - "
                     f"<a href='{map_url}' target='_blank'>view on map</a>")
    status_html.value = score_text()
    for b in choice_buttons:
        b.disabled = True
    next_button.disabled = False


for b in choice_buttons:
    b.on_click(answer)
next_button.on_click(new_question)

display(widgets.VBox([out_fig, widgets.HBox(choice_buttons, layout=widgets.Layout(flex_flow='row wrap')),
                      out_msg, widgets.HBox([next_button, status_html])]))
new_question()

In [ ]:
# your confusion table so far: rows are the true class, columns your guess
if history:
    display(pd.crosstab(pd.Series([t for t, _ in history], name='true'), pd.Series([g for _, g in history], name='guess')))